# HOPE-CAD v2.2: Kaggle W2/W6 Run All

**Before Run All**

1. In Kaggle Settings, select a GPU accelerator with at least 12 GiB VRAM (T4 is fine), start the session, and enable Internet so missing Python packages can install.
2. Keep the source bundle and `mvtec_prefix_features_final.pt` inputs already attached. The source may be an extracted folder; the feature bundle is a separate `.pt` file.
3. To run W2, also attach `mvtec_anomaly_detection.tar.xz` and the ViT-B/8 checkpoint whose SHA256 is `2c4839589adbbf701573dd155e1814f8adb145cd4d235bb86a93269369e111ec`. Cell 2 safely extracts the MVTec archive into `/kaggle/working` and finds the 15 categories automatically. If the archive/checkpoint is absent, W2 is marked pending and W6 can still proceed from the cached feature bundle.

Run All verifies source hashes, runs the repository tests, validates the 50/20/15/15 role split without reading probe/test data, optionally extracts a fresh prefix cache and tests a two-task checkpoint/resume smoke for W2, then runs real-feature W6 selection for M, C0, C1 and C2. It never runs W7 anomaly-test evaluation or uses test metrics for selection.

**Completion boundary:** a passed W2 smoke verifies clean-session extraction and task-boundary checkpoint resume on two normal tasks; it does not replace a full 15-task interrupted-run acceptance. W6 is complete only when all four selection artifacts report `selected` and required gates pass. The CLI retains fit checkpoints but does not yet resume an interrupted full search. The notebook packages outputs under `/kaggle/working/hope_cad_v2_2_w2_w6_evidence.zip`; these are engineering artifacts, not reportable CAD performance.

In [1]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import tarfile
import zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
REPO_ROOT = WORK_ROOT / 'hope_cad_v2_2'
SOURCE_BUNDLE_OVERRIDE = None  # Optional path to source ZIP or extracted source folder.
FEATURE_BUNDLE_OVERRIDE = None
MVTEC_ROOT_OVERRIDE = None  # Optional root containing bottle/, cable/, ..., zipper/.
VIT_B8_CHECKPOINT_OVERRIDE = None  # Optional exact ImageNet-21k ViT-B/8 checkpoint path.

EXPECTED_SOURCE_NAME = 'hope_cad_v2_2_source_bundle.zip'
EXPECTED_FEATURE_NAME = 'mvtec_prefix_features_final.pt'
EXPECTED_MVTEC_ARCHIVE = 'mvtec_anomaly_detection.tar.xz'
CANONICAL_MVTEC = ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']

def unique_input_file(override, exact_name, fallback_pattern, label):
    if override:
        path = Path(override)
        if not path.is_file() or not path.resolve().is_relative_to(INPUT_ROOT.resolve()):
            raise FileNotFoundError(f'{label} override must point to a file under {INPUT_ROOT}: {path}')
        return path
    exact = [p for p in INPUT_ROOT.rglob(exact_name) if p.is_file()]
    matches = exact or [p for p in INPUT_ROOT.rglob('*') if p.is_file() and re.search(fallback_pattern, p.name, re.I)]
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {label}; found {len(matches)}: {matches}. Attach the correct Kaggle Dataset or set its override in this cell.')
    return matches[0]

def find_source_input(override=None):
    if override:
        path = Path(override)
        if not path.resolve().is_relative_to(INPUT_ROOT.resolve()):
            raise FileNotFoundError(f'Source override must be under {INPUT_ROOT}: {path}')
        if path.is_file() and path.suffix.lower() == '.zip':
            return path, None
        if path.is_dir() and (path / 'bundle_manifest.json').is_file():
            return None, path
        raise FileNotFoundError(f'Source override must be a ZIP or extracted bundle folder: {path}')
    archives = [p for p in INPUT_ROOT.rglob(EXPECTED_SOURCE_NAME) if p.is_file()]
    if len(archives) == 1:
        return archives[0], None
    if len(archives) > 1:
        raise RuntimeError(f'Multiple source ZIPs found: {archives}; set SOURCE_BUNDLE_OVERRIDE.')
    trees = [p.parent for p in INPUT_ROOT.rglob('bundle_manifest.json')
             if (p.parent / 'pixi.toml').is_file() and (p.parent / 'models').is_dir()]
    if len(trees) == 1:
        return None, trees[0]
    raise RuntimeError(f'Expected one source ZIP or extracted bundle tree; found {len(trees)} trees: {trees}.')

def looks_like_mvtec_root(path):
    path = Path(path)
    return all((path / category / 'train' / 'good').is_dir() for category in CANONICAL_MVTEC)

def find_extracted_mvtec_roots(base):
    base = Path(base)
    return {p.parents[2] for p in base.rglob('bottle/train/good')
            if p.is_dir() and looks_like_mvtec_root(p.parents[2])}

SOURCE_ZIP, SOURCE_TREE = find_source_input(SOURCE_BUNDLE_OVERRIDE)
FEATURE_BUNDLE = unique_input_file(FEATURE_BUNDLE_OVERRIDE, EXPECTED_FEATURE_NAME, r'prefix.*features.*\.pt$', 'W6 feature bundle')

MVTEC_ROOT = Path(MVTEC_ROOT_OVERRIDE) if MVTEC_ROOT_OVERRIDE else None
if MVTEC_ROOT is not None and not looks_like_mvtec_root(MVTEC_ROOT):
    raise FileNotFoundError(f'MVTec root does not contain all 15 canonical categories: {MVTEC_ROOT}')
if MVTEC_ROOT is None:
    existing_roots = find_extracted_mvtec_roots(INPUT_ROOT)
    if len(existing_roots) == 1:
        MVTEC_ROOT = existing_roots.pop()
    elif len(existing_roots) > 1:
        raise RuntimeError(f'Multiple extracted MVTec roots found: {sorted(map(str, existing_roots))}')
    else:
        archives = [p for p in INPUT_ROOT.rglob(EXPECTED_MVTEC_ARCHIVE) if p.is_file()]
        if len(archives) > 1:
            raise RuntimeError(f'Multiple MVTec archives found: {archives}')
        if len(archives) == 1:
            extract_dir = WORK_ROOT / 'mvtec_ad_extracted'
            extracted_roots = find_extracted_mvtec_roots(extract_dir) if extract_dir.exists() else set()
            if len(extracted_roots) == 1:
                MVTEC_ROOT = extracted_roots.pop()
            elif len(extracted_roots) > 1:
                raise RuntimeError(f'Multiple MVTec roots in extraction directory: {sorted(map(str, extracted_roots))}')
            else:
                if extract_dir.exists():
                    shutil.rmtree(extract_dir)
                extract_dir.mkdir(parents=True)
                with tarfile.open(archives[0], mode='r:xz') as tar:
                    members = tar.getmembers()
                    for member in members:
                        relative = PurePosixPath(member.name)
                        if (relative.is_absolute() or '..' in relative.parts or chr(92) in member.name
                                or member.issym() or member.islnk() or member.isdev() or member.isfifo()):
                            raise RuntimeError(f'Unsafe path in MVTec archive: {member.name}')
                    tar.extractall(extract_dir, members=members, filter='data')
                extracted_roots = find_extracted_mvtec_roots(extract_dir)
                if len(extracted_roots) != 1:
                    raise RuntimeError(f'Expected one MVTec root after extraction; found {len(extracted_roots)}: {sorted(map(str, extracted_roots))}')
                MVTEC_ROOT = extracted_roots.pop()

VIT_CHECKPOINT = Path(VIT_B8_CHECKPOINT_OVERRIDE) if VIT_B8_CHECKPOINT_OVERRIDE else None
if VIT_CHECKPOINT is not None and (not VIT_CHECKPOINT.is_file() or not VIT_CHECKPOINT.resolve().is_relative_to(INPUT_ROOT.resolve())):
    raise FileNotFoundError(f'ViT checkpoint override must point under {INPUT_ROOT}: {VIT_CHECKPOINT}')
if VIT_CHECKPOINT is None:
    candidates = [p for p in INPUT_ROOT.rglob('*') if p.is_file()
                  and p.suffix.lower() in {'.safetensors', '.pt', '.pth'}
                  and re.search(r'vit|patch8|in21k', p.name, re.I)]
    if len(candidates) == 1:
        VIT_CHECKPOINT = candidates[0]
    elif len(candidates) > 1:
        raise RuntimeError(f'Multiple ViT-like checkpoints found: {candidates}; set VIT_B8_CHECKPOINT_OVERRIDE.')

print('Source ZIP:', SOURCE_ZIP if SOURCE_ZIP else 'not present')
print('Extracted source tree:', SOURCE_TREE if SOURCE_TREE else 'not present')
print('Feature bundle:', FEATURE_BUNDLE)
print('MVTec root:', MVTEC_ROOT if MVTEC_ROOT else 'not attached; W2 will be pending')
print('ViT checkpoint:', VIT_CHECKPOINT if VIT_CHECKPOINT else 'not attached; W2 will be pending')

In [2]:
from pathlib import PurePosixPath
import stat

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
REPO_ROOT.mkdir(parents=True)

if SOURCE_ZIP is not None:
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        for info in archive.infolist():
            name = info.filename
            relative = PurePosixPath(name)
            mode = info.external_attr >> 16
            if relative.is_absolute() or '..' in relative.parts or chr(92) in name or stat.S_ISLNK(mode):
                raise RuntimeError(f'Unsafe source archive entry: {name}')
            destination = (REPO_ROOT / Path(*relative.parts)).resolve()
            if not destination.is_relative_to(REPO_ROOT.resolve()):
                raise RuntimeError(f'Archive path escapes extraction root: {name}')
        archive.extractall(REPO_ROOT)
    SOURCE_FORMAT = 'zip'
    SOURCE_BUNDLE_SHA256 = hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest()
else:
    source_manifest_path = SOURCE_TREE / 'bundle_manifest.json'
    source_manifest_raw = source_manifest_path.read_bytes()
    source_manifest = json.loads(source_manifest_raw)
    SOURCE_FORMAT = 'kaggle_extracted_tree'
    SOURCE_BUNDLE_SHA256 = hashlib.sha256(json.dumps(source_manifest['files'], sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    for relative, expected in source_manifest['files'].items():
        source_path = (SOURCE_TREE / relative).resolve()
        if not source_path.is_relative_to(SOURCE_TREE.resolve()) or not source_path.is_file():
            raise RuntimeError(f'Invalid/missing extracted source path: {relative}')
        if hashlib.sha256(source_path.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Source SHA256 mismatch: {relative}')
        destination = REPO_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination)
    shutil.copy2(source_manifest_path, REPO_ROOT / 'bundle_manifest.json')

manifest_path = REPO_ROOT / 'bundle_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError('Source input has no bundle_manifest.json at its root.')
source_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if source_manifest.get('schema') != 'stage1_source_bundle_v1':
    raise RuntimeError(f"Unexpected source bundle schema: {source_manifest.get('schema')}")
hash_failures = []
for relative, expected in source_manifest['files'].items():
    path = (REPO_ROOT / relative).resolve()
    if not path.is_relative_to(REPO_ROOT.resolve()) or not path.is_file():
        hash_failures.append(relative)
    elif hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        hash_failures.append(relative)
if hash_failures:
    raise RuntimeError(f'Source SHA256 mismatch/missing files: {hash_failures}')

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Verified source files:', len(source_manifest['files']))
print('Source format:', SOURCE_FORMAT)
print('Source identity SHA256:', SOURCE_BUNDLE_SHA256)
print('Source branch/commit:', source_manifest.get('origin', {}).get('branch'), source_manifest.get('origin', {}).get('source_commit'))
print('Working tree:', REPO_ROOT)

In [3]:
EXPECTED_VIT_B8_SHA256 = '2c4839589adbbf701573dd155e1814f8adb145cd4d235bb86a93269369e111ec'
VIT_CHECKPOINT_SHA256 = None
if VIT_CHECKPOINT is not None:
    digest = hashlib.sha256()
    with VIT_CHECKPOINT.open('rb') as checkpoint_stream:
        for chunk in iter(lambda: checkpoint_stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    VIT_CHECKPOINT_SHA256 = digest.hexdigest()
    print('ViT checkpoint SHA256:', VIT_CHECKPOINT_SHA256)
    if VIT_CHECKPOINT_SHA256.lower() != EXPECTED_VIT_B8_SHA256:
        raise RuntimeError(
            'Wrong ViT checkpoint. Expected SHA256 ' + EXPECTED_VIT_B8_SHA256 +
            ' but got ' + VIT_CHECKPOINT_SHA256 + '. W2 is stopped to prevent a mismatched backbone.')
else:
    print('No ViT checkpoint attached; W2 will be pending, W6 may still use the validated feature bundle.')

In [4]:
import importlib.util
import importlib.metadata
import torch

missing = [name for name, module in [('timm', 'timm'), ('safetensors', 'safetensors')] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.30', 'safetensors==0.7.0'])

torch.set_num_threads(2)
torch.set_float32_matmul_precision('highest')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

CUDA_OK = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_OK else None
GPU_BYTES = torch.cuda.get_device_properties(0).total_memory if CUDA_OK else 0
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', CUDA_OK)
print('GPU:', GPU_NAME)
print('GPU memory bytes:', GPU_BYTES)
print('timm:', importlib.metadata.version('timm'))
print('safetensors:', importlib.metadata.version('safetensors'))

test_command = [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests/hope_cad_v2', '-v']
test_log = WORK_ROOT / 'hope_cad_v2_2_unit_tests.log'
with test_log.open('w', encoding='utf-8') as log:
    test_result = subprocess.run(test_command, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, text=True)
test_text = test_log.read_text(encoding='utf-8', errors='replace')
print('\n'.join(test_text.splitlines()[-12:]))
if test_result.returncode != 0:
    raise RuntimeError(f'HOPE test suite failed with exit code {test_result.returncode}; full log: {test_log}')

In [5]:
from scripts.hope_cad_v2.calibrate_features import bundle_provider
from dataset.benchmark_protocol_v2 import file_hash

bundle_sha256 = file_hash(FEATURE_BUNDLE)
feature_bundle = torch.load(FEATURE_BUNDLE, map_location='cpu', weights_only=False)
feature_data = bundle_provider(feature_bundle, 'cpu')
role_counts = {name: len(rows) for name, rows in feature_bundle['role_manifest']['roles'].items()}
expected_counts = {'meta_support': 50, 'meta_query': 20, 'normal_validation': 15, 'normal_probe': 15}
if role_counts != expected_counts:
    raise RuntimeError(f'Unexpected frozen prefix split: {role_counts}')
if len(feature_bundle['images']) != 85:
    raise RuntimeError(f"Expected 85 cached support/query/validation images; got {len(feature_bundle['images'])}")
if feature_data.reads['normal_probe'] != 0:
    raise RuntimeError('Feature validation unexpectedly read normal_probe')
if feature_bundle['feature_identity']['design_id'] != 'hope_cad_v3_masked_teacher_cms':
    raise RuntimeError('Feature bundle design_id does not match frozen v3 design')

FEATURE_INFO = {
    'path': str(FEATURE_BUNDLE),
    'sha256': bundle_sha256,
    'bytes': FEATURE_BUNDLE.stat().st_size,
    'design_id': feature_bundle['feature_identity']['design_id'],
    'backbone_sha256': feature_bundle['feature_identity']['backbone_sha256'],
    'role_counts': role_counts,
    'feature_images': len(feature_bundle['images']),
    'probe_reads': feature_data.reads['normal_probe'],
    'official_test_accessed': False,
}
print(json.dumps(FEATURE_INFO, indent=2))
del feature_data, feature_bundle

In [6]:
W2_RESULT = {'status': 'pending_missing_assets', 'resume_export_acceptance': 'pending', 'official_test_accessed': False}
if MVTEC_ROOT is None or VIT_CHECKPOINT is None:
    W2_RESULT['reason'] = 'Attach the complete MVTec root and exact ViT-B/8 ImageNet-21k checkpoint to run W2 clean-session extraction and task-boundary resume smoke.'
else:
    from datetime import datetime, timezone
    from torch.utils.data import DataLoader
    from dataset.benchmark_manifest_v1 import canonical_digest
    from dataset.benchmark_protocol_v1 import _ImageDataset
    from dataset.benchmark_protocol_v2 import calibration_roles
    from models.hope_cad_v2.predictor import HOPEFeaturePredictor
    from models.hope_cad_v2.visual_block import VisualBlockConfig
    from models.hope_cad_v2.benchmark_adapter import HOPEBenchmarkAdapterV3
    from training.benchmark_artifacts_v1 import BenchmarkArtifacts
    from training.benchmark_config_v1 import load_configs
    from training.benchmark_engine_v1 import BenchmarkEngineV1
    from training.hope_outer_learning_v2 import export_initialization
    checkpoint_sha256 = file_hash(VIT_CHECKPOINT)
    os.environ['MVTEC_ROOT'] = str(MVTEC_ROOT)
    os.environ['VIT_B8_CHECKPOINT'] = str(VIT_CHECKPOINT)
    os.environ['VIT_B8_SHA256'] = checkpoint_sha256
    lock_dir = WORK_ROOT / 'cadic_protocol_lock'
    w2_feature_output = WORK_ROOT / 'w2_clean_prefix_features.pt'
    prepare_lock = subprocess.run(
        [sys.executable, '-B', 'scripts/hope_cad_v2/run_matched_cadic.py',
         '--phase', 'prepare', '--run-dir', str(lock_dir)],
        cwd=REPO_ROOT, text=True, capture_output=True)
    W2_RESULT.update({'checkpoint_sha256': checkpoint_sha256,
                      'lock_exit_code': prepare_lock.returncode,
                      'lock_stdout': prepare_lock.stdout[-4000:],
                      'lock_stderr': prepare_lock.stderr[-4000:]})
    if prepare_lock.returncode == 0:
        extraction = subprocess.run(
            [sys.executable, '-B', 'scripts/hope_cad_v2/prepare_prefix_features.py',
             '--run-dir', str(lock_dir), '--output', str(w2_feature_output), '--device', 'cuda'],
            cwd=REPO_ROOT, text=True, capture_output=True)
        W2_RESULT.update({'extraction_exit_code': extraction.returncode,
                          'extraction_stdout': extraction.stdout[-4000:],
                          'extraction_stderr': extraction.stderr[-4000:]})
        if extraction.returncode == 0:
            cache_report = json.loads(w2_feature_output.with_suffix('.json').read_text(encoding='utf-8'))
            manifest = json.loads((lock_dir / 'manifest_train.json').read_text(encoding='utf-8'))
            protocol, method = load_configs(
                'conf/benchmarks/protocols/mvtec_1x15_prefix_cal_v2.yaml',
                'conf/benchmarks/methods/cadic_matched_prefix_v2.yaml')
            protocol_identity = {'protocol_id': protocol['id'], 'manifest_digest': manifest['digest'],
                                'baseline_lock_digest': json.loads((lock_dir / 'baseline_lock.json').read_text())['digest']}
            model = HOPEFeaturePredictor(VisualBlockConfig(), method='M', beta=1e-3)
            initialization = export_initialization(model, {
                'roles_digest': calibration_roles(manifest)['digest'],
                'schedule_digest': 'w2-two-task-resume-smoke',
                'source_sha256': cache_report['source']['source_sha256'],
                'spec_sha256': {k: v for k, v in cache_report['source']['source_sha256'].items() if k.startswith('docs/')},
                'feature_identity': cache_report['feature_identity'],
            })
            initialization_path = WORK_ROOT / 'w2_smoke_initialization.pt'
            torch.save(initialization, initialization_path)
            initialization = torch.load(initialization_path, map_location='cpu', weights_only=False)
            def make_adapter():
                return HOPEBenchmarkAdapterV3(
                    initialization, checkpoint_path=VIT_CHECKPOINT, checkpoint_sha256=checkpoint_sha256,
                    checkpoint_identity=method['extractor']['checkpoint_identity'], device='cuda',
                    protocol_identity=protocol_identity)
            task_rows = {}
            for row in manifest['entries']:
                task_rows.setdefault(row['task_name'], []).append(row)
            resume_tasks = ['bottle', 'cable']
            if any(name not in task_rows for name in resume_tasks):
                raise RuntimeError('W2 resume smoke cannot find the first two canonical normal tasks in the manifest')
            class TwoTaskNormalSmoke:
                task_names = resume_tasks
                def build_train_loader(self, task_id, max_images=None):
                    row = task_rows[self.task_names[task_id]][0]
                    dataset = _ImageDataset(MVTEC_ROOT, [row], 224, method['preprocessing']['mean'],
                                            method['preprocessing']['std'], training=True)
                    return DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
            resume_dir = WORK_ROOT / ('w2_resume_smoke_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
            first_adapter = make_adapter()
            artifacts = BenchmarkArtifacts(resume_dir)
            engine = BenchmarkEngineV1(protocol, first_adapter, artifacts, 'cuda', fail_on_state_mutation=True)
            engine.train(TwoTaskNormalSmoke(), max_tasks=1, resume=False)
            first_adapter.close()
            del engine, artifacts, first_adapter
            torch.cuda.empty_cache()
            resumed_adapter = make_adapter()
            resumed_engine = BenchmarkEngineV1(protocol, resumed_adapter, BenchmarkArtifacts(resume_dir),
                                                'cuda', fail_on_state_mutation=True)
            resumed_engine.train(TwoTaskNormalSmoke(), max_tasks=2, resume=True)
            resumed_state = torch.load(resume_dir / 'states' / 'final.pt', map_location='cpu', weights_only=False)
            resumed_adapter.load_state_dict(resumed_state)
            resume_passed = (resumed_adapter.session.cursor == 2 and
                             resumed_adapter.session.state.clock == 2 and
                             resumed_adapter.session.state.commits == (2, 0))
            resumed_adapter.close()
            if not resume_passed:
                raise RuntimeError('W2 task-boundary resume did not restore and continue the CMS clock')
            W2_RESULT.update({
                'status': 'clean_session_extractor_and_task_resume_smoke_passed',
                'real_prefix_feature_bundle_sha256': cache_report['bundle_sha256'],
                'role_counts': cache_report['role_counts'],
                'probe_features_extracted': cache_report['probe_features_extracted'],
                'official_test_accessed': False,
                'two_task_boundary_resume': 'passed',
                'resume_run_dir': str(resume_dir),
                'initialization_artifact': str(initialization_path),
                'resume_export_acceptance': 'two-task checkpoint export/resume smoke passed; full 15-task recovery remains unverified',
                'random_unselected_initialization': True,
                'reportable': False,
            })
        else:
            W2_RESULT.update({'status': 'real_extractor_smoke_failed'})
    else:
        W2_RESULT.update({'status': 'protocol_or_asset_preflight_failed'})

w2_report = WORK_ROOT / 'w2_clean_session_preflight.json'
w2_report.write_text(json.dumps(W2_RESULT, indent=2) + '\n', encoding='utf-8')
print(json.dumps(W2_RESULT, indent=2))

In [7]:
from datetime import datetime, timezone

W6_RUN_ROOT = WORK_ROOT / ('w6_run_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
W6_RUN_ROOT.mkdir(parents=True, exist_ok=False)
W6_METHODS = ('M', 'C0', 'C1', 'C2')
W6_RESULTS = []
resource_reason = None
if not CUDA_OK:
    resource_reason = 'CUDA is unavailable.'
elif GPU_BYTES < 12 * 1024**3:
    resource_reason = f'GPU has {GPU_BYTES} bytes; canonical W6 requires at least 12 GiB.'

if resource_reason:
    gate = {'status': 'pending_external_gpu', 'reportable': False, 'required_gpu_bytes': 12 * 1024**3,
            'available_gpu_bytes': GPU_BYTES, 'reason': resource_reason}
    (W6_RUN_ROOT / 'resource_gate.json').write_text(json.dumps(gate, indent=2) + '\n', encoding='utf-8')
    W6_RESULTS.append(gate)
    print(json.dumps(gate, indent=2))
else:
    for method in W6_METHODS:
        method_dir = W6_RUN_ROOT / f'method_{method}'
        log_path = W6_RUN_ROOT / f'method_{method}.log'
        command = [sys.executable, '-B', 'scripts/hope_cad_v2/calibrate_features.py',
                   '--bundle', str(FEATURE_BUNDLE), '--output-dir', str(method_dir),
                   '--method', method, '--device', 'cuda']
        print('Starting', method, 'with output', method_dir)
        with log_path.open('w', encoding='utf-8') as log:
            result = subprocess.run(command, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, text=True)
        selection_path = method_dir / 'selection.json'
        selection = json.loads(selection_path.read_text(encoding='utf-8')) if selection_path.is_file() else None
        entry = {'method': method, 'exit_code': result.returncode, 'status': selection.get('status') if selection else 'missing_selection_artifact',
                 'selected_beta': selection.get('selected_beta') if selection else None,
                 'selected_step': selection.get('selected_step') if selection else None,
                 'output_dir': str(method_dir), 'log': str(log_path),
                 'deployment_initialization_exists': (method_dir / 'deployment_initialization.pt').is_file()}
        W6_RESULTS.append(entry)
        print(json.dumps(entry, indent=2))
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

W6_SUMMARY = {'status': 'pending_external_gpu' if resource_reason else
              ('all_methods_selected' if all(r.get('status') == 'selected' for r in W6_RESULTS) else 'incomplete_or_selection_failed'),
              'reportable': False, 'selection_rule_id': 'prefix_cms_stress_v3',
              'feature_bundle_sha256': bundle_sha256, 'gpu': GPU_NAME, 'gpu_bytes': GPU_BYTES,
              'methods': W6_RESULTS,
              'full_search_resume': 'not implemented in CLI; fit-level checkpoints are retained',
              'W7_test_evaluation_run': False}
(W6_RUN_ROOT / 'w6_summary.json').write_text(json.dumps(W6_SUMMARY, indent=2) + '\n', encoding='utf-8')
print(json.dumps(W6_SUMMARY, indent=2))

In [ ]:
FEATURE_VALIDATION_REPORT = WORK_ROOT / 'feature_bundle_validation.json'
FEATURE_VALIDATION_REPORT.write_text(json.dumps(FEATURE_INFO, indent=2) + '\n', encoding='utf-8')
W2_REPORT = WORK_ROOT / 'w2_clean_session_preflight.json'
HANDOFF = WORK_ROOT / 'hope_cad_v2_2_w2_w6_evidence.zip'
HANDOFF_INFO = {
    'source_format': SOURCE_FORMAT,
    'source_bundle_sha256': SOURCE_BUNDLE_SHA256,
    'source_manifest_origin': source_manifest.get('origin', {}),
    'feature_bundle': FEATURE_INFO,
    'w2': W2_RESULT,
    'w6': W6_SUMMARY,
    'unit_test_log': str(test_log),
    'W7_run': False,
    'anomaly_test_metrics_used_for_selection': False,
    'known_limitations': ['W2 resume evidence is a two-task boundary smoke; full 15-task interrupted recovery is not run.',
                          'W6 CLI saves fit checkpoints but has no full-search resume option.',
                          'W7 MVTec anomaly-test evaluation is intentionally not run.'],
}
handoff_json = WORK_ROOT / 'hope_cad_v2_2_handoff_summary.json'
handoff_json.write_text(json.dumps(HANDOFF_INFO, indent=2) + '\n', encoding='utf-8')

with zipfile.ZipFile(HANDOFF, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(handoff_json, handoff_json.name)
    archive.write(test_log, test_log.name)
    archive.write(FEATURE_VALIDATION_REPORT, FEATURE_VALIDATION_REPORT.name)
    if W2_REPORT.is_file():
        archive.write(W2_REPORT, W2_REPORT.name)
    if W2_RESULT.get('resume_run_dir'):
        resume_root = Path(W2_RESULT['resume_run_dir'])
        for path in resume_root.rglob('*'):
            if path.is_file():
                archive.write(path, Path('w2_resume') / path.relative_to(resume_root))
    if W2_RESULT.get('initialization_artifact'):
        init_path = Path(W2_RESULT['initialization_artifact'])
        if init_path.is_file():
            archive.write(init_path, Path('w2_resume') / init_path.name)
    for path in W6_RUN_ROOT.rglob('*'):
        if path.is_file():
            archive.write(path, Path('w6') / path.relative_to(W6_RUN_ROOT))

print('Evidence archive:', HANDOFF)
print('Evidence ZIP SHA256:', hashlib.sha256(HANDOFF.read_bytes()).hexdigest())
print('W2 status:', W2_RESULT['status'], '| resume/export:', W2_RESULT['resume_export_acceptance'])
print('W6 status:', W6_SUMMARY['status'])
print('W7 executed:', HANDOFF_INFO['W7_run'])
if (W6_RUN_ROOT / 'method_M' / 'deployment_initialization.pt').is_file():
    print('Selected M initialization:', W6_RUN_ROOT / 'method_M' / 'deployment_initialization.pt')

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
import zipfile

ARTIFACT_ZIP = WORK_ROOT / "hope_cad_v2_2_all_artifacts.zip"
EXCLUDED_DIR_NAME = "mvtec_ad_extracted"

with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in WORK_ROOT.rglob("*"):
        if not path.is_file():
            continue

        # Không đưa thư mục MVTec đã giải nén vào archive
        if EXCLUDED_DIR_NAME in path.relative_to(WORK_ROOT).parts:
            continue

        # Không tự đưa file ZIP đang tạo vào chính nó
        if path.resolve() == ARTIFACT_ZIP.resolve():
            continue

        archive.write(path, path.relative_to(WORK_ROOT))

print(f"Artifact ZIP: {ARTIFACT_ZIP}")
print(f"Size: {ARTIFACT_ZIP.stat().st_size / 1024**2:.2f} MB")
display(FileLink(str(ARTIFACT_ZIP), result_html_prefix="Download all artifacts: "))